Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = "Nicole Paraschiv"
COLLABORATORS = ""

-- [E006] Not Found Error: cmd1.sc:1:13 ----------------------------------------
1 |val res1_0 = NAME = "Nicole Paraschiv"
  |             ^^^^
  |             Not found: NAME
  |
  | longer explanation available when compiling with `-explain`
-- [E006] Not Found Error: cmd1.sc:2:13 ----------------------------------------
2 |val res1_1 = COLLABORATORS = ""
  |             ^^^^^^^^^^^^^
  |             Not found: COLLABORATORS
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

---

# CSCI 3155 Assignment 6 : Let Bindings and Scopes

This assignment asks you to write scala programs. 

**Restrictions** apply to each problem in terms of forbidden Scala features and API functions. Please read them carefully and ask for clarifications from the course staff over Piazza or during office hours if unsure.

Note: `???` indicates that there is a missing function or code fragment that needs to be filled in. In scala, 
it is also a macro that throws a `NotImplemented` exception. Make sure that you remove the `???` and replace it with the answer. 

Use the test cases provided to test them. You are also encouraged to write your own test cases to help debug your work. However, please delete any extra cells you may have created lest they break our autograder.

**Very Important:** Please run the cell that defines the functions `passed` and `testWithMessage` below whenever you restart the notebook.

In [1]:
// FIRST RUN THIS CELL EVERY TIME YOU START THE NOTEBOOK

def passed(points: Int): Float =  {
    require(points >=0)
    if (points == 1) print(s"\n*** Tests Passed (1 point) ***\n")
    else print(s"\n*** Tests Passed ($points points) ***\n")
    points.toFloat
}

def time[T](block: => T) : T = {
    /* block is a call by name parameter -- it will not be evaluated when the method is called
       but will be evaluated inside the method */
    val t1 = System.nanoTime()
    val res = block 
    val t2 = System.nanoTime()
    println(s"\t Elapsed time: ${(t2 - t1) / 1e6d} ms")
    res
}

def testWithMessage[T](v1: => T, expected: T, testID: String, debug: Boolean = true) = { 
    println(s"Running Test $testID"); 
    val v1_res =  time(v1) 
    if (debug){
        println(s"\t Expected: $expected, your code returned: $v1_res")
    }
    assert (v1_res == expected, s"Test $testID FAILED.")
    println("\t Passed!")
}


defined function passed
defined function time
defined function testWithMessage

## Part A: Multiple Simultaneous Let Bindings (20 points)

In class, we studied let bindings that assigned a "single" identifier to a "single" expression. Here, we will extend lettuce with multiple let bindings at the same time:

## Example 1 
~~~
let (x, y, z) = (10, 25.6, 30.3) in 
   x - y * z
~~~

The program computes `10 - 25.6 * 30.3`.

## Example 2

 In multi-let binding, we treat all the assignments as happening "simultaneously". For instance, the program 

~~~
let (x, y, z) = (10, x, y - x) in 
  x - y * z
~~~

is disallowed since neither `x` nor `y` are in scope in the right hand side of the multi-let binding. Example 2 above produces an `error` value since `x` and `y` are out of scope on the right hand side of the assignment.

## Example 3

~~~
let x = 15 in 
  let (x, y, z) = (x*x, -10 *x, -2*x) in 
     x + y + z
~~~

Note that the usage `x*x`, `-10*x` and `-2*x` refer back to `let x = 15` definition. However, the usages `x+y+z` refer to the result of the "multi-let" binding. Verify that this program will evaluate to "45".

## Grammar of Lettuce

Let us extend a minimalistic subset of Lettuce by adding a `MultiLet` statement as shown below.
$$\newcommand\Expr{\mathbf{Expr}}$$

$$\begin{array}{rcll}
  \Expr & \Rightarrow & \text{Const}(\mathbf{Double}) \\
  & | & \text{Ident}(\mathbf{String}) \\
  & | & \text{Plus}(\mathbf{Expr}, \mathbf{Expr}) \\
  & | & \text{Mult}(\mathbf{Expr}, \mathbf{Expr})\\
  & | & \text{Let}(\mathbf{Ident}, \mathbf{Expr}, \mathbf{Expr}) \\
  & | & \text{MultiLet}(\mathbf{Ident}*, \mathbf{Expr}*, \mathbf{Expr}) & \leftarrow\ \text{ let (x1, .., xn) = (e1, ...,en) in e } \\
  & & & \text{Note: Number of identifiers n must match number of expressions n, or else evaluate to error }\\
  \end{array}$$
  
The scala definitions are given below.

In [2]:
sealed trait Expr
case class Const(d: Double) extends Expr
case class Ident(s: String) extends Expr
case class Plus(e1: Expr, e2: Expr) extends Expr
case class Mult(e1: Expr, e2: Expr) extends Expr 
case class Let(id: String, e1: Expr, e2: Expr) extends Expr
case class MultiLet(id: List[String], eList: List[Expr], e2: Expr) extends Expr

sealed trait Value
case class NumValue(f: Double) extends Value
case object Error extends Value /* -- Do not return Error -- simply throw an new IllegalArgumentException whenever you encounter an erroneous case --*/

type Environment = Map[String, Value]

defined trait Expr
defined class Const
defined class Ident
defined class Plus
defined class Mult
defined class Let
defined class MultiLet
defined trait Value
defined class NumValue
defined object Error
defined type Environment

## Semantics for MultiLet

$$\newcommand\semrule[3]{\begin{array}{c} #1 \\ \hline #2 \\ \end{array}(\text{#3})} $$

Let us write down the semantic rules for a multilet statement:

$$\newcommand\eval{\textit{eval}}$$
$$\semrule{ \eval(\texttt{ei}, \texttt{env})= v_i,\ v_i \not= \mathbf{error}, \text{for}\ i = 1, \ldots, n,\ \texttt{newenv} = env \circ \{ \texttt{x1} \mapsto v_1, \ldots, \texttt{xn} \mapsto v_n \} }{ \eval( \texttt{MultiLet([x1,..,xn], [e1,...,en], eBody), env}) = \eval(\texttt{eBody, newenv})}{multilet-ok}$$

The semantic rule above tells you to 
  - Evaluate each of the expressions from `e1`, ..., `en` under the environment `env`.
  - Next, if all the expressions above evaluated without an error, it tells you to update the map `env` by binding each `xi` to $v_i$, the result of evaluating `ei`. You can use the Scala Map "++" operator to achieve this in one shot.
  
  ~~~
  val m1 : Map[String, Int] = Map( "x" -> 10, "y" -> 20, "z" -> 30 )
  val m2 : Map[String, Int] = Map( "w" -> 40, "l" -> 50, "z" -> 60)
  val m3 = m1 ++ m2 // Join the two maps together and obtain a new map. z will map to 60
  ~~~
  - Finally, you should evaluate `eBody` under the new environment created.

For convenience, we write a single "generic" semantic rule that shows that if some argument `ej` evaluates to an error, the whole expression is erroneous.

$$\semrule{ j \leq n, \eval(\texttt{ei}, \texttt{env})= v_i,\ v_i \not= \mathbf{error}, \text{for}\ i = 1, \ldots, j-1,\ \ eval(\texttt{ej}, \texttt{env})= \mathbf{error} }{ \eval( \texttt{MultiLet([x1,..,xn], [e1,...,en], eBody), env}) = \mathbf{error}}{multilet-err-j}$$

### Interpreter for MultiLet Statements

Implement an interpreter for the lettuce language with `multi-let` statements. Your interpreter does not need to "propagate" error: instead you should throw an `IllegalArgumentException` whenever an error is encountered. 

### Style Guide

Use of var/while/for loops in your solution below is disallowed.

Complete the function `evalA` below.

In [3]:

def evalA(e: Expr, env: Environment): Value = {
    
    e match {
        case Const(f) => NumValue(f)
        case Ident(x) => { 
            if (env.contains(x)) { 
                env(x)
            } else {
                throw new IllegalArgumentException("Not found identifier")
            }
        }
        case Plus(e1, e2) => {
            val v1 = evalA(e1, env)
            val v2 = evalA(e2, env)
            (v1, v2) match {
                case (NumValue(f1), NumValue(f2)) => NumValue(f1 + f2)
                case _ => throw new IllegalArgumentException("plus failed")
            }
        }
        case Mult(e1, e2) => {
            val v1 = evalA(e1, env)
            val v2 = evalA(e2, env)
            (v1, v2) match {
                case (NumValue(f1), NumValue(f2)) => NumValue(f1 * f2)
                case _ => throw new IllegalArgumentException("mult failed")
            }
        }
        case Let(x, e1, e2) => {
            // YOUR CODE HERE
            val v1 = evalA(e1, env)
            val newEnv = env + (x -> v1)
            evalA(e2, newEnv)
        }
        case MultiLet(xList, eList, e2) => {
            // YOUR CODE HERE
            if(xList.length != eList.length){
                throw new IllegalArgumentException("MultiLet Exception")
            }
            val values = eList.map(e => evalA(e,env))
            val bind = xList.zip(values).toMap
            val newEnv = env ++ bind
            evalA(e2,newEnv)
        }
    }
   
}

defined function evalA

In [4]:
//BEGIN TEST
/*
 let (x, y) = (10, 20) in 
    let x = y in 
      x +  x * y
*/
val x = Ident("x")
val y = Ident("y")
val let1 = Let("x", y, Plus(x, Mult(x, y)) )
val mlet1 = MultiLet( List("x", "y"), List(Const(10.0), Const(20.0)), let1)
val v = evalA(mlet1, Map.empty)
assert(v == NumValue(420.0), s"Test 1 failed expected: NumValue(420.0), obtained $v")

passed(7)
//END TEST


*** Tests Passed (7 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
let1: Let = Let(
  id = "x",
  e1 = Ident(s = "y"),
  e2 = Plus(
    e1 = Ident(s = "x"),
    e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
  )
)
mlet1: MultiLet = MultiLet(
  id = List("x", "y"),
  eList = List(Const(d = 10.0), Const(d = 20.0)),
  e2 = Let(
    id = "x",
    e1 = Ident(s = "y"),
    e2 = Plus(
      e1 = Ident(s = "x"),
      e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
    )
  )
)
v: Value = NumValue(f = 420.0)
res4_6: Float = 7.0F

In [5]:
//BEGIN TEST
/*
 let (x, y) = (10, x) in 
    let x = y in 
      x +  x * y
*/
val x = Ident("x")
val y = Ident("y")
val let1 = Let("x", y, Plus(x, Mult(x, y)) )
val mlet1 = MultiLet( List("x", "y"), List(Const(10.0), x), let1)
try {
    val v = evalA(mlet1, Map.empty)
    assert(false, "Test 2 failed -- your code should detect a usage of x that is out of scope")
} catch {
    case e:IllegalArgumentException => { println("Illegal argument exception caught -- as expected!!") }
    case _ => {println("Wrong type of exception thrown")}
}

passed(8)
//END TEST

Illegal argument exception caught -- as expected!!

*** Tests Passed (8 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
let1: Let = Let(
  id = "x",
  e1 = Ident(s = "y"),
  e2 = Plus(
    e1 = Ident(s = "x"),
    e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
  )
)
mlet1: MultiLet = MultiLet(
  id = List("x", "y"),
  eList = List(Const(d = 10.0), Ident(s = "x")),
  e2 = Let(
    id = "x",
    e1 = Ident(s = "y"),
    e2 = Plus(
      e1 = Ident(s = "x"),
      e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
    )
  )
)
res5_5: Float = 8.0F

In [6]:
//BEGIN TEST
/*
let (x, y, z, w) = (10, 10, 10, 20 ) in 
  let () = () in 
    let w = w in 
       x *( y + w )
*/

val x = Ident("x")
val y = Ident("y")
val z = Ident("z")
val w = Ident("w")
val ten = Const(10.0)
val twenty = Const(20.0)
val innerLet2 = Let("w", w, Mult(x, Plus(y, w)))
val multiLet1 = MultiLet(Nil, Nil, innerLet2)
val e = MultiLet(List("x","y","z","w"), List(ten, ten, ten, twenty), multiLet1)
val v = evalA(e, Map.empty)
assert(v == NumValue(300.0), "Test2 Failed -- expected value NumValue(300.0), obtained value $v")

passed(10)
//END TEST


*** Tests Passed (10 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
z: Ident = Ident(s = "z")
w: Ident = Ident(s = "w")
ten: Const = Const(d = 10.0)
twenty: Const = Const(d = 20.0)
innerLet2: Let = Let(
  id = "w",
  e1 = Ident(s = "w"),
  e2 = Mult(
    e1 = Ident(s = "x"),
    e2 = Plus(e1 = Ident(s = "y"), e2 = Ident(s = "w"))
  )
)
multiLet1: MultiLet = MultiLet(
  id = List(),
  eList = List(),
  e2 = Let(
    id = "w",
    e1 = Ident(s = "w"),
    e2 = Mult(
      e1 = Ident(s = "x"),
      e2 = Plus(e1 = Ident(s = "y"), e2 = Ident(s = "w"))
    )
  )
)
e: MultiLet = MultiLet(
  id = List("x", "y", "z", "w"),
  eList = List(
    Const(d = 10.0),
    Const(d = 10.0),
    Const(d = 10.0),
    Const(d = 20.0)
  ),
  e2 = MultiLet(
    id = List(),
    eList = List(),
    e2 = Let(
      id = "w",
      e1 = Ident(s = "w"),
      e2 = Mult(
        e1 = Ident(s = "x"),
        e2 = Plus(e1 = Ident(s = "y"), e2 = Ident(s = "w"))
      )
    )
  )
)
v: Value = NumValue(f = 300.0)
res6_11: Float = 10.0

## Part B : Different Semantics for MultiLet (18 points)

$$\newcommand\semrule[3]{\begin{array}{c} #1 \\ \hline #2 \\ \end{array}(\text{#3})} $$

Let us write down a different semantic rule for a multilet statement.
In this semantics, we effectively translate a mult-let staement into a nested series of let statements.

```
let (x1, ..., xn) = (e1, ..., en) in 
  e
```
in this semantics is the same as

```
  let x1 = e1 in 
     let x2 = e2 in 
        ... 
          let xn = en in
             e 
```

Think about why this is different from the previous semantics.


$$\newcommand\eval{\textit{eval}}$$
$$\semrule{ {n \geq 1},\  \eval(\texttt{e1}, \texttt{env})= v_1,\ v_1 \not= \mathbf{error},\ \texttt{newenv} = env \circ \{ \texttt{x1} \mapsto v_1\} }{ \eval( \texttt{MultiLet([x1,..,xn], [e1,...,en], eBody), env}) = \eval(\texttt{MultiLet([x2,..., xn], [e2, ..., en], eBody)}, \texttt{newenv})}{multilet-non-empty}$$

The semantic rule above tells you that
  - If $n \geq 1$, i.e., your multilet statement has one more bindings happening,
  - Evaluate the very first expression `e1`under the environment `env`.
  - Next, if this is evaluated without an error, it tells you to update the map `env` by binding  `x1` to $v_1$, the result of evaluating `e1` to obtain a new environment called `newenv`.
  - Finally, you should evaluate the "remaining" multilet binding that binds `[x2,...,xn]` to `[e2,...,en]`, respectively under the environment `newenv` from previous step. Note that we are providing a recursive definition of the `MultiLet` statement.

We will now provide the last rule for an "empty" multilet statement:

$$\semrule{}{ \eval( \texttt{MultiLet([], [], eBody), env}) = \eval(\texttt{eBody}, env)}{multilet-empty}$$

We will let you fill in the error rules. The interpreter will raise an `IllegalArgumentException` whenever an error is encountered.

### Interpreter for MultiLet Statements

Implement an interpreter for the lettuce language with `multi-let` statements. Your interpreter does not need to "propagate" error: instead you should throw an `IllegalArgumentException` whenever an error is encountered. 

Let's call this function `evalB`. 

### Style Guide

Use of var/while/for loops in your solution below is disallowed.

In [7]:


def evalB(e: Expr, env: Environment): Value = {
    
    e match {
        case Const(f) => NumValue(f)
        case Ident(x) => { 
            if (env.contains(x)) { 
                env(x)
            } else {
                throw new IllegalArgumentException("Not found identifier")
            }
        }
        case Plus(e1, e2) => {
            val v1 = evalB(e1, env)
            val v2 = evalB(e2, env)
            (v1, v2) match {
                case (NumValue(f1), NumValue(f2)) => NumValue(f1 + f2)
                case _ => throw new IllegalArgumentException("plus failed")
            }
        }
        case Mult(e1, e2) => {
            val v1 = evalB(e1, env)
            val v2 = evalB(e2, env)
            (v1, v2) match {
                case (NumValue(f1), NumValue(f2)) => NumValue(f1 * f2)
                case _ => throw new IllegalArgumentException("mult failed")
            }
        }
        case Let(x, e1, e2) => {
            // YOUR CODE HERE
            val v1 = evalB(e1, env)
            val newEnv = env + (x -> v1)
            evalB(e2, newEnv)
        }
        case MultiLet(xList, eList, eBody) => {
            // YOUR CODE HERE
            (xList, eList) match {
                case (Nil, Nil) => {
                    evalB(eBody, env)
                }
                case(xHead::xTail, eHead::eTail) => {
                    val vHead = evalB(eHead,env)
                    val newEnv = env + (xHead -> vHead)
                    evalB(MultiLet(xTail, eTail, eBody), newEnv)
                }
                case _ => throw new IllegalArgumentException("MultiLet")
            }
        }
    }
   
}

defined function evalB

In [8]:
//BEGIN TEST
/*
 let (x, y) = (10, 20) in 
    let x = y in 
      x +  x * y
*/
val x = Ident("x")
val y = Ident("y")
val let1 = Let("x", y, Plus(x, Mult(x, y)) )
val mlet1 = MultiLet( List("x", "y"), List(Const(10.0), Const(20.0)), let1)
val v = evalB(mlet1, Map.empty)
assert(v == NumValue(420.0), s"Test 1 failed expected: NumValue(420.0), obtained $v")

passed(6)
//END TEST


*** Tests Passed (6 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
let1: Let = Let(
  id = "x",
  e1 = Ident(s = "y"),
  e2 = Plus(
    e1 = Ident(s = "x"),
    e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
  )
)
mlet1: MultiLet = MultiLet(
  id = List("x", "y"),
  eList = List(Const(d = 10.0), Const(d = 20.0)),
  e2 = Let(
    id = "x",
    e1 = Ident(s = "y"),
    e2 = Plus(
      e1 = Ident(s = "x"),
      e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
    )
  )
)
v: Value = NumValue(f = 420.0)
res8_6: Float = 6.0F

In [9]:
//BEGIN TEST
/*
 let (x, y) = (10, x) in 
    let x = y in 
      x +  x * y
*/
val x = Ident("x")
val y = Ident("y")
val let1 = Let("x", y, Plus(x, Mult(x, y)) )
val mlet1 = MultiLet( List("x", "y"), List(Const(10.0), x), let1)
val v = evalB(mlet1, Map.empty)
assert(v == NumValue(110.0), s"Test Failed: expected NumValue(110.0), your code returns $v")


passed(6)
//END TEST


*** Tests Passed (6 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
let1: Let = Let(
  id = "x",
  e1 = Ident(s = "y"),
  e2 = Plus(
    e1 = Ident(s = "x"),
    e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
  )
)
mlet1: MultiLet = MultiLet(
  id = List("x", "y"),
  eList = List(Const(d = 10.0), Ident(s = "x")),
  e2 = Let(
    id = "x",
    e1 = Ident(s = "y"),
    e2 = Plus(
      e1 = Ident(s = "x"),
      e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
    )
  )
)
v: Value = NumValue(f = 110.0)
res9_6: Float = 6.0F

In [10]:
//BEGIN TEST
/*
let (x, y, z, w) = (10, 10, 10, 20 ) in 
  let () = () in 
    let w = w in 
       x *( y + w )
*/

val x = Ident("x")
val y = Ident("y")
val z = Ident("z")
val w = Ident("w")
val ten = Const(10.0)
val twenty = Const(20.0)
val innerLet2 = Let("w", w, Mult(x, Plus(y, w)))
val multiLet1 = MultiLet(Nil, Nil, innerLet2)
val e = MultiLet(List("x","y","z","w"), List(ten, ten, ten, twenty), multiLet1)
val v = evalB(e, Map.empty)
assert(v == NumValue(300.0), "Test2 Failed -- expected value NumValue(300.0), obtained value $v")

passed(6)
//END TEST


*** Tests Passed (6 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
z: Ident = Ident(s = "z")
w: Ident = Ident(s = "w")
ten: Const = Const(d = 10.0)
twenty: Const = Const(d = 20.0)
innerLet2: Let = Let(
  id = "w",
  e1 = Ident(s = "w"),
  e2 = Mult(
    e1 = Ident(s = "x"),
    e2 = Plus(e1 = Ident(s = "y"), e2 = Ident(s = "w"))
  )
)
multiLet1: MultiLet = MultiLet(
  id = List(),
  eList = List(),
  e2 = Let(
    id = "w",
    e1 = Ident(s = "w"),
    e2 = Mult(
      e1 = Ident(s = "x"),
      e2 = Plus(e1 = Ident(s = "y"), e2 = Ident(s = "w"))
    )
  )
)
e: MultiLet = MultiLet(
  id = List("x", "y", "z", "w"),
  eList = List(
    Const(d = 10.0),
    Const(d = 10.0),
    Const(d = 10.0),
    Const(d = 20.0)
  ),
  e2 = MultiLet(
    id = List(),
    eList = List(),
    e2 = Let(
      id = "w",
      e1 = Ident(s = "w"),
      e2 = Mult(
        e1 = Ident(s = "x"),
        e2 = Plus(e1 = Ident(s = "y"), e2 = Ident(s = "w"))
      )
    )
  )
)
v: Value = NumValue(f = 300.0)
res10_11: Float = 6.0

## Part C (7 points) 

Write a program (an expression) using multi-let that gives different answers in the semantics from part A when compared to the semantics from part B. For instance, your program from one of the semantics may give you an error, whereas the other program may give you a proper numerical value.

In [13]:
val e = { //TODO: Write your expression as a multilet expression using abstract syntax.
    // YOUR CODE HERE
    MultiLet(List("x", "y"), List(Const(10.0), x), let1)
}

e: MultiLet = MultiLet(
  id = List("x", "y"),
  eList = List(Const(d = 10.0), Ident(s = "x")),
  e2 = Let(
    id = "x",
    e1 = Ident(s = "y"),
    e2 = Plus(
      e1 = Ident(s = "x"),
      e2 = Mult(e1 = Ident(s = "x"), e2 = Ident(s = "y"))
    )
  )
)

In [14]:
val v1 = try { 
        evalA(e, Map.empty)
} catch { 
    case _ => Error
}
println(s"Part A semantics yields $v1")

val v2 = try{ 
            evalB(e, Map.empty)
} catch {
    case _ => Error 
}
println(s"Part B semantics yields $v2")

val matches = (v1, v2) match {
    case (Error, Error) => assert(false, "Both semantics yield the same value Error")
    case (NumValue(_), Error) => true
    case (Error, NumValue(_)) => true
    case (NumValue(i1), NumValue(i2)) if i1 == i2 => assert(false, s"Both semantics yield the same value NumValue($i1) = NumValue($i2)")
    case (_, _) => true
}

println("Passed the test")

passed(7)

Part A semantics yields Error
Part B semantics yields NumValue(110.0)
Passed the test

*** Tests Passed (7 points) ***


v1: Value = Error
v2: Value = NumValue(f = 110.0)
matches: Boolean = true
res14_6: Float = 7.0F

## That's All Folks!